# Volume 3. NEMO-Style Toy Parser

Question: how does the maintained assembly-calculus parser connect word categories, roles, and word order in a controlled example?

This notebook uses `assembly_calculus.NemoParser`, not the optional GPU-heavy `neural_assemblies.nemo` stack. The point is to show the package surface that currently runs in the default CPU environment.

In [ ]:
from neural_assemblies.core.brain import Brain
from neural_assemblies.assembly_calculus import overlap
from neural_assemblies.assembly_calculus.parser import NemoParser

Create a small parser. The vocabulary and training set are deliberately tiny so the mechanics are visible.

In [ ]:
brain = Brain(p=0.05, save_winners=True, seed=42, engine="numpy_sparse")
parser = NemoParser(brain, n=3_000, k=50, beta=0.1, rounds=5)
parser.setup_areas()

for noun in ["dog", "cat"]:
    parser.register_word(noun, "noun", f"vis_{noun}")
parser.register_word("chases", "verb", "mot_chases")

Train three separate pieces: lexical category, thematic role, and word order.

In [ ]:
training_sentences = [["dog", "chases", "cat"]]

parser.train_lexicon()
parser.train_roles(training_sentences)
parser.train_word_order(training_sentences)

Parse the training sentence and inspect the returned dictionary. This is a controlled smoke example, not a general language benchmark.

In [ ]:
result = parser.parse(["dog", "chases", "cat"])
print("categories:", result["categories"])
print("roles:", result["roles"])

The internal role lexicons are ordinary assembly snapshots. You can inspect them with the same tools used in the foundations volume.

In [ ]:
agent_dog = parser.role_lexicons["ROLE_AGENT"]["dog"]
patient_cat = parser.role_lexicons["ROLE_PATIENT"]["cat"]
print("agent area:", agent_dog.area)
print("patient area:", patient_cat.area)
print("cross-role overlap:", f"{overlap(agent_dog, patient_cat):.3f}")

What to notice: this parser is compositional in the code structure. It separates category learning, role binding, and sequence memory so each part can be tested and debugged.